# Chapter 19: Desktop Audio Player and Frequency Visualizer

Companion notebook for *Python by Curiosity*. Every code listing from the chapter is
here, in the order it appears in the book, under the same title. Run the cells from
top to bottom: some listings use variables or functions defined in earlier cells.

The same listings are also available as separate scripts in `codes/listings/ch19_music_player/`.

**These listings open desktop windows (Tkinter).** Run the notebook on your own
computer (Jupyter or VS Code), not in an online service such as Google Colab. Each
cell waits until you close its window; close it to move on to the next cell.

In [ ]:
# Run the listings from the repository root so that paths such as
# data/grades.csv and figures/trophy.png work.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())

### Code 19.1: Scanning media files from a folder

In [ ]:
import os

def load_music_folder(folder_path):
    valid_extensions = ('.mp3', '.wav', '.ogg', '.flac')
    playlist_files = []
    
    for file in os.listdir(folder_path):
        if file.lower().endswith(valid_extensions):
            full_path = os.path.join(folder_path, file)
            playlist_files.append((file, full_path))
            
    return playlist_files

### Code 19.2: Playlist queue navigation logic

In [ ]:
import random

class PlaylistManager:
    def __init__(self, track_list):
        self.tracks = track_list
        self.current_idx = 0
        self.is_shuffle = False

    def next_track(self):
        if not self.tracks:
            return None
        if self.is_shuffle:
            self.current_idx = random.randint(0, len(self.tracks) - 1)
        else:
            self.current_idx = (self.current_idx + 1) % len(self.tracks)
        return self.tracks[self.current_idx]

    def prev_track(self):
        if not self.tracks:
            return None
        self.current_idx = (self.current_idx - 1) % len(self.tracks)
        return self.tracks[self.current_idx]

### Code 19.3: Track position scrubbing callback

In [ ]:
def on_scrub(self, value):
    self.position = float(value)          # Scale passes a string
    self.lbl_elapsed.config(text=format_time(self.position))

### Code 19.4: Canvas spectrum visualizer animation loop

In [ ]:
import random

def animate_spectrum(canvas, bar_ids):
    for bar_id in bar_ids:
        new_height = random.randint(10, 85)
        x1, _, x2, _ = canvas.coords(bar_id)
        canvas.coords(bar_id, x1, 95 - new_height, x2, 95)   # move the top edge
    # Run again after 50 ms (20 frames per second)
    canvas.after(50, lambda: animate_spectrum(canvas, bar_ids))

### Code 19.5: Desktop Music Player & Spectrum Visualizer

In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog
import os
import random

TRACK_SECONDS = 210          # pretend length of every track (3:30)


def format_time(seconds):
    return f"{int(seconds // 60):02d}:{int(seconds % 60):02d}"


class MusicPlayerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Desktop Music Player & Spectrum Visualizer")
        self.root.geometry("700x440")
        self.root.configure(bg="#1e272c")

        # --- Application state (the "model") ---
        self.tracks = []          # list of (file name, full path)
        self.current_idx = 0
        self.is_playing = False
        self.is_shuffle = False
        self.speed = 1.0
        self.position = 0.0       # seconds into the current track

        self.build_ui()
        self.tick()               # start the timer/animation loop

    # ------------------------------------------------------------------
    def build_ui(self):
        tk.Label(self.root, text="DESKTOP AUDIO PLAYER & SPECTRUM VISUALIZER",
                 font=("Arial", 11, "bold"), bg="#101820", fg="#00ffcc",
                 pady=6).pack(fill="x")

        body = tk.Frame(self.root, bg="#1e272c", padx=10, pady=10)
        body.pack(fill="both", expand=True)

        # Left panel: playlist
        left = tk.Frame(body, bg="#263238", padx=8, pady=8, bd=1, relief="ridge")
        left.pack(side="left", fill="y", padx=5)
        tk.Button(left, text="Open Folder...", command=self.open_folder).pack(fill="x")
        self.listbox = tk.Listbox(left, width=24, bg="#1c2833", fg="#00ffcc",
                                  selectbackground="#27ae60")
        self.listbox.pack(fill="both", expand=True, pady=5)
        self.listbox.bind("<<ListboxSelect>>", self.on_select)

        # Right panel: now playing, visualizer, scrub bar, controls
        right = tk.Frame(body, bg="#263238", padx=10, pady=10, bd=1, relief="ridge")
        right.pack(side="right", fill="both", expand=True, padx=5)

        self.lbl_now = tk.Label(right, text="No folder loaded", font=("Arial", 10, "bold"),
                                bg="#263238", fg="#f1c40f")
        self.lbl_now.pack(anchor="w")

        self.canvas = tk.Canvas(right, height=100, bg="#101820", highlightthickness=0)
        self.canvas.pack(fill="x", pady=8)
        self.bars = []
        for i in range(16):
            x = 10 + 17 * i
            self.bars.append(self.canvas.create_rectangle(
                x, 90, x + 14, 95, fill="#27ae60", outline=""))

        time_row = tk.Frame(right, bg="#263238")
        time_row.pack(fill="x")
        self.lbl_elapsed = tk.Label(time_row, text="00:00", bg="#263238", fg="#00ffcc")
        self.lbl_elapsed.pack(side="left")
        self.scrub = ttk.Scale(time_row, from_=0, to=TRACK_SECONDS, command=self.on_scrub)
        self.scrub.pack(side="left", fill="x", expand=True, padx=8)
        tk.Label(time_row, text=format_time(TRACK_SECONDS), bg="#263238",
                 fg="#00ffcc").pack(side="right")

        buttons = tk.Frame(right, bg="#263238")
        buttons.pack(pady=8)
        tk.Button(buttons, text="|< Prev", width=7, command=self.prev_track).pack(side="left", padx=2)
        self.btn_play = tk.Button(buttons, text="Play", width=7, command=self.toggle_play)
        self.btn_play.pack(side="left", padx=2)
        tk.Button(buttons, text="Next >|", width=7, command=self.next_track).pack(side="left", padx=2)
        self.btn_shuffle = tk.Button(buttons, text="Shuffle: OFF", command=self.toggle_shuffle)
        self.btn_shuffle.pack(side="left", padx=2)
        self.btn_speed = tk.Button(buttons, text="Speed: 1.0x", command=self.toggle_speed)
        self.btn_speed.pack(side="left", padx=2)

        # Equalizer sliders (interface only: see the text)
        eq = tk.Frame(right, bg="#263238")
        eq.pack(fill="x")
        for band in ["31Hz", "250Hz", "1kHz", "4kHz", "16kHz"]:
            box = tk.Frame(eq, bg="#263238")
            box.pack(side="left", expand=True)
            tk.Scale(box, from_=10, to=-10, length=50, showvalue=0,
                     bg="#263238", highlightthickness=0).pack()
            tk.Label(box, text=band, font=("Arial", 7), bg="#263238", fg="#00ffcc").pack()

        self.lbl_status = tk.Label(self.root, text="Status: open a folder to begin",
                                   anchor="w", relief="sunken", bg="#101820", fg="#00ffcc")
        self.lbl_status.pack(side="bottom", fill="x")

    # ------------------------------------------------------------------
    def open_folder(self):
        folder = filedialog.askdirectory(title="Choose a music folder")
        if not folder:                       # user pressed Cancel
            return
        valid = ('.mp3', '.wav', '.ogg', '.flac')
        self.tracks = [(f, os.path.join(folder, f))
                       for f in sorted(os.listdir(folder))
                       if f.lower().endswith(valid)]
        self.listbox.delete(0, tk.END)
        for name, _ in self.tracks:
            self.listbox.insert(tk.END, name)
        self.current_idx = 0
        self.load_track()

    def load_track(self):
        self.position = 0.0
        self.scrub.set(0)
        if not self.tracks:
            self.lbl_now.config(text="No audio files in that folder")
        else:
            self.listbox.selection_clear(0, tk.END)
            self.listbox.selection_set(self.current_idx)
            self.lbl_now.config(text=f"Track: {self.tracks[self.current_idx][0]}")
        self.update_status()

    def on_select(self, event):
        sel = self.listbox.curselection()
        if not sel:
            return
        self.current_idx = sel[0]
        self.load_track()

    def next_track(self):
        if not self.tracks:
            return
        if self.is_shuffle:
            self.current_idx = random.randint(0, len(self.tracks) - 1)
        else:
            self.current_idx = (self.current_idx + 1) % len(self.tracks)
        self.load_track()

    def prev_track(self):
        if not self.tracks:
            return
        self.current_idx = (self.current_idx - 1) % len(self.tracks)
        self.load_track()

    def toggle_play(self):
        if not self.tracks:
            return
        self.is_playing = not self.is_playing
        self.btn_play.config(text="Pause" if self.is_playing else "Play")
        self.update_status()

    def toggle_shuffle(self):
        self.is_shuffle = not self.is_shuffle
        self.btn_shuffle.config(text=f"Shuffle: {'ON' if self.is_shuffle else 'OFF'}")
        self.update_status()

    def toggle_speed(self):
        speeds = [1.0, 1.5, 2.0]
        self.speed = speeds[(speeds.index(self.speed) + 1) % len(speeds)]
        self.btn_speed.config(text=f"Speed: {self.speed}x")

    def on_scrub(self, value):
        self.position = float(value)         # Scale passes a string
        self.lbl_elapsed.config(text=format_time(self.position))

    def update_status(self):
        if not self.tracks:
            text = "Status: no tracks loaded"
        else:
            state = "Playing" if self.is_playing else "Paused"
            shuffle = "ON" if self.is_shuffle else "OFF"
            text = (f"Status: {state} track {self.current_idx + 1} of "
                    f"{len(self.tracks)} | Shuffle: {shuffle}")
        self.lbl_status.config(text=text)

    # ------------------------------------------------------------------
    def tick(self):
        """Runs every 100 ms: advance the clock and animate the bars."""
        if self.is_playing:
            self.position += 0.1 * self.speed
            if self.position >= TRACK_SECONDS:
                self.next_track()
            self.scrub.set(self.position)
            for bar in self.bars:
                h = random.randint(12, 88)   # simulated spectrum
                x1, _, x2, _ = self.canvas.coords(bar)
                self.canvas.coords(bar, x1, 95 - h, x2, 95)
        self.root.after(100, self.tick)


if __name__ == "__main__":
    root = tk.Tk()
    app = MusicPlayerApp(root)
    root.mainloop()

### Adding Real Sound (Optional)

*Shown in the book as a fragment or a deliberate mistake; on its own it stops with ModuleNotFoundError.* The code is shown here for reference:

```python
import pygame
pygame.mixer.init()                      # once, in __init__

# in load_track():
pygame.mixer.music.load(self.tracks[self.current_idx][1])

# in toggle_play():
if self.is_playing:
    pygame.mixer.music.play(start=self.position)
else:
    pygame.mixer.music.pause()
```

### Worked Example 19.1: Audio Waveform Canvas Painter

In [ ]:
import tkinter as tk
import numpy as np

class WaveformPainterApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Audio Waveform Canvas Painter")
        self.root.geometry("540x360")
        self.root.configure(bg="#1a1a2e")
        
        # Header Controls
        ctrl = tk.Frame(root, bg="#16213e", padx=10, pady=8)
        ctrl.pack(side="top", fill="x")
        
        tk.Label(ctrl, text="Freq 1 (Hz):", fg="white", bg="#16213e", font=("Arial", 9)).pack(side="left")
        self.scale_f1 = tk.Scale(ctrl, from_=1, to=10, orient="horizontal", bg="#16213e", fg="white",
                                 highlightthickness=0, command=self.redraw)
        self.scale_f1.set(2)
        self.scale_f1.pack(side="left", padx=5)
        
        tk.Label(ctrl, text="Freq 2 (Hz):", fg="white", bg="#16213e", font=("Arial", 9)).pack(side="left", padx=(10, 0))
        self.scale_f2 = tk.Scale(ctrl, from_=1, to=20, orient="horizontal", bg="#16213e", fg="white",
                                 highlightthickness=0, command=self.redraw)
        self.scale_f2.set(6)
        self.scale_f2.pack(side="left", padx=5)
        
        # Waveform Canvas
        self.canvas = tk.Canvas(root, bg="#0f3460", highlightthickness=0)
        self.canvas.pack(fill="both", expand=True, padx=10, pady=10)
        
        self.redraw()

    def redraw(self, event=None):
        self.canvas.delete("all")
        w = self.canvas.winfo_width()
        h = self.canvas.winfo_height()
        if w <= 1:
            w, h = 520, 260
            
        mid_y = h / 2.0
        # Draw central zero-axis
        self.canvas.create_line(0, mid_y, w, mid_y, fill="#533483", dash=(4, 4))
        
        f1 = self.scale_f1.get()
        f2 = self.scale_f2.get()
        t = np.linspace(0, 1, w)
        # Superposition of two harmonic tones
        signal = 0.6 * np.sin(2 * np.pi * f1 * t) + 0.3 * np.sin(2 * np.pi * f2 * t)
        
        # Convert mathematical coordinates to canvas pixel points
        points = []
        for x_pixel, val in enumerate(signal):
            y_pixel = mid_y - (val * (h * 0.4))
            points.extend([x_pixel, y_pixel])
            
        if len(points) >= 4:
            self.canvas.create_line(points, fill="#00fff5", width=2, smooth=True)

if __name__ == "__main__":
    root = tk.Tk()
    app = WaveformPainterApp(root)
    root.mainloop()

### Worked Example 19.2: Playlist Duration Calculator with Metadata

In [ ]:
import tkinter as tk

def format_duration(seconds):
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins:02d}:{secs:02d}"

class PlaylistSummaryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Album Playlist Metadata Summary")
        self.root.geometry("420x280")
        self.root.configure(padx=15, pady=15)
        
        # Album track database: (Title, Duration in Seconds)
        self.album = [
            ("Track 1: Morning Light", 214),
            ("Track 2: Quantum Cascade", 185),
            ("Track 3: Midnight Drift", 342),
            ("Track 4: Solar Wind", 198),
            ("Track 5: Velvet Horizon", 276)
        ]
        
        tk.Label(root, text="ALBUM METADATA & DURATION AUDIT", font=("Arial", 11, "bold"), fg="#2c3e50").pack(pady=(0, 10))
        
        # Track Listbox
        self.lb = tk.Listbox(root, font=("Courier", 9), height=6)
        for title, dur in self.album:
            self.lb.insert(tk.END, f"{title:<28} [{format_duration(dur)}]")
        self.lb.pack(fill="x", pady=5)
        
        # Calculate Album Statistics
        total_sec = sum(dur for _, dur in self.album)
        longest = max(self.album, key=lambda item: item[1])
        shortest = min(self.album, key=lambda item: item[1])
        
        summary_text = (
            f"Total Tracks: {len(self.album)}\n"
            f"Total Album Playtime: {format_duration(total_sec)} ({total_sec} sec)\n"
            f"Longest Track: {longest[0]} ({format_duration(longest[1])})\n"
            f"Shortest Track: {shortest[0]} ({format_duration(shortest[1])})"
        )
        
        lbl_info = tk.Label(root, text=summary_text, font=("Arial", 10), justify="left",
                            bg="#ecf0f1", padx=10, pady=8, relief="groove", bd=1)
        lbl_info.pack(fill="both", expand=True, pady=10)

if __name__ == "__main__":
    root = tk.Tk()
    app = PlaylistSummaryApp(root)
    root.mainloop()